# Lesson 3: Conditional Routing — sum vs multiply vs divide

## Objective
Use `add_conditional_edges` to dynamically **route** execution to the correct arithmetic node based on the `operation` field in state, running only ONE node per call.

## Limitation of Lesson 2
- ❌ All three nodes always ran — even when only one was needed
- ❌ No branching — the graph was a straight sequential line
- ❌ Wasted computation on every call

## What is NEW in Lesson 3?
- ✅ `add_conditional_edges(source, router_fn, path_map)` — the core routing API
- ✅ **Router function** pattern: reads state → returns a string key
- ✅ **path_map** dict: maps router return values to node names
- ✅ Diverging graph topology: one node branches into multiple possible paths
- ✅ `Literal` type hint to document valid router return values

## Theory: Conditional Edges

Without conditional routing, graphs are pipelines.  
With conditional routing, graphs become **decision trees and state machines**.

```
                    ┌── "add"      ──→ add_node      ──→ END
START → router_node ┼── "multiply" ──→ multiply_node ──→ END
                    └── "divide"   ──→ divide_node   ──→ END
```

`add_conditional_edges(source, router_fn, path_map)`:
- **source**: which node triggers routing AFTER it runs
- **router_fn(state) → str**: reads state, returns a string key
- **path_map**: maps string keys to next node names

⚠️ The router function **must NOT modify state** — it only reads.


In [ ]:
# ── Cell 1: Imports ─────────────────────────────────────────────────────────
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Optional, Literal
from IPython.display import Image, display

# Literal["add", "multiply", "divide"] tells Python exactly what strings are valid
# This makes your router type-safe and self-documenting

print("✓ Imports successful")
print("  Literal: restricts type to a set of string values")


In [ ]:
# ── Cell 2: State Schema ────────────────────────────────────────────────────
class State(TypedDict):
    a: int
    b: int
    operation: str          # "add", "multiply", or "divide"
    result: Optional[float] # Single result field — ONE operation runs

print("✓ State defined")
print("  'operation' drives routing; only one node runs per invoke()")


In [ ]:
# ── Cell 3: Arithmetic Nodes ────────────────────────────────────────────────
def add_node(state: State) -> dict:
    res = state["a"] + state["b"]
    print(f"  [add]      {state['a']} + {state['b']} = {res}")
    return {"result": float(res)}

def multiply_node(state: State) -> dict:
    res = state["a"] * state["b"]
    print(f"  [multiply] {state['a']} × {state['b']} = {res}")
    return {"result": float(res)}

def divide_node(state: State) -> dict:
    if state["b"] == 0:
        print("  [divide]   ERROR: division by zero → result = None")
        return {"result": None}
    res = state["a"] / state["b"]
    print(f"  [divide]   {state['a']} ÷ {state['b']} = {res:.4f}")
    return {"result": res}

print("✓ Three nodes defined (only one will run per invocation)")


## The Router Function — Core New Concept

The router function is a **pure read function**:
- Input: current state
- Output: a string key that must match a key in `path_map`
- Side effects: NONE (never modifies state)

```python
def router(state: State) -> Literal["add", "multiply", "divide"]:
    return state["operation"]   # Just read and return
```

`Literal["add", "multiply", "divide"]` is a type annotation that:
1. Documents what values are valid
2. Enables IDE autocomplete
3. Lets LangGraph validate the path_map at compile time


In [ ]:
# ── Cell 4: Router Function ─────────────────────────────────────────────────
# Literal type hint: only these three strings are valid return values
# LangGraph uses the returned string to look up the next node in path_map

def router(state: State) -> Literal["add", "multiply", "divide"]:
    op = state["operation"]
    print(f"  [router]   Inspecting operation='{op}' → routing to '{op}'")
    return op   # This string MUST match a key in the path_map below

print("✓ Router defined")
print("  Router: reads state['operation'] → returns routing key")


In [ ]:
# ── Cell 5: Pass-Through Router Node ────────────────────────────────────────
# We need a node to trigger conditional routing from START
# This lightweight node does nothing except trigger the conditional edge

def router_node(state: State) -> dict:
    return {}  # No state changes — just a trigger for routing

print("✓ Router node (pass-through) defined")


In [ ]:
# ── Cell 6: Build the Conditional Graph ─────────────────────────────────────
builder = StateGraph(State)

# Register all nodes
builder.add_node("router_node", router_node)
builder.add_node("add",         add_node)
builder.add_node("multiply",    multiply_node)
builder.add_node("divide",      divide_node)

# Entry: START always goes to router_node
builder.add_edge(START, "router_node")

# ─── THE KEY LINE ─────────────────────────────────────────────────────────
# add_conditional_edges(source, router_fn, path_map)
#   source     = "router_node"  ← after this node runs, call the router
#   router_fn  = router          ← reads state, returns "add"/"multiply"/"divide"
#   path_map   = {...}           ← maps returned string to next node
builder.add_conditional_edges(
    "router_node",    # Source node
    router,           # Router function (state → string)
    {                 # path_map: router return value → node name
        "add":      "add",
        "multiply": "multiply",
        "divide":   "divide",
    }
)

# All operation nodes lead to END
builder.add_edge("add",      END)
builder.add_edge("multiply", END)
builder.add_edge("divide",   END)

graph = builder.compile()
print("✓ Conditional graph compiled")
print("  Graph branches: one of add/multiply/divide runs per call")


In [ ]:
# ── Cell 7: Visualize ───────────────────────────────────────────────────────
# You should see: __start__ → router_node → (3 branches) → __end__
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
# ── Cell 8: Test routing to 'add' ────────────────────────────────────────────
print("="*50)
print("Test: operation = 'add'")
print("="*50)
out = graph.invoke({"a": 15, "b": 5, "operation": "add", "result": None})
print(f"  Result: {out['result']}")


In [ ]:
# ── Cell 9: Test routing to 'multiply' ──────────────────────────────────────
print("="*50)
print("Test: operation = 'multiply'")
print("="*50)
out = graph.invoke({"a": 15, "b": 5, "operation": "multiply", "result": None})
print(f"  Result: {out['result']}")


In [ ]:
# ── Cell 10: Test routing to 'divide' ───────────────────────────────────────
print("="*50)
print("Test: operation = 'divide'")
print("="*50)
out = graph.invoke({"a": 15, "b": 5, "operation": "divide", "result": None})
print(f"  Result: {out['result']}")


In [ ]:
# ── Cell 11: Compare — Only One Node Runs ────────────────────────────────────
print("Demonstrating that ONLY the routed node runs:")
print()
for op in ["add", "multiply", "divide"]:
    print(f"--- operation='{op}' ---")
    out = graph.invoke({"a": 6, "b": 3, "operation": op, "result": None})
    print(f"    result = {out['result']}")
    print()


## Alternative: Conditional Edge from START (Cleaner)

You can skip the pass-through node entirely:
```python
builder.add_conditional_edges(START, router, {"add":"add", ...})
```

This is the pattern used in Lessons 6+ when the LLM IS the router.

## Key Line Summary

| Line | What it does |
|------|-------------|
| `add_conditional_edges("router_node", router, {...})` | After router_node runs, call `router(state)` and use result to pick next node |
| `router(state) -> Literal[...]` | Pure function: reads state, returns routing key |
| `{"add": "add", ...}` | path_map: maps router return to node name |
| `Literal[...]` | Type-safe documentation of valid routing keys |

## Summary

| | Lesson 2 | Lesson 3 |
|--|----------|----------|
| Nodes executed | All 3 always | Only 1 (chosen at runtime) |
| Graph shape | Linear | Branching |
| Routing | None | `add_conditional_edges` |

## Limitations → What Lesson 4 Solves
- ❌ `result` field is overwritten each time — no history
- ❌ If two nodes write to the same field simultaneously, the last one wins
- ❌ **Lesson 4**: Reducers accumulate results safely using `Annotated`
